# Signal Filtering 2 — Basic Smoothing (moving average / EWMA / median)
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

A noisy **weight / load-cell** signal: fill/rest/empty cycles with quantization dither, agitation/sloshing, and washing-fluid spikes. numpy, scipy, pandas and matplotlib are pre-installed in Colab.

> Runs on a synthetic signal that mimics a real load cell. **To use your own export**, replace the load cell with `pd.read_csv("your_export.csv", ...)` (two columns: timestamp, value).

## Objectives
By the end you will be able to:
- Apply moving-average, EWMA and median filters.
- See the window / span / kernel trade-off (smoothness vs lag).
- See why a median filter rejects spikes a moving average smears.

## Load & define the filters
These three are the everyday smoothers (adapted from the weight-filter toolkit).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
s = pd.read_csv(DATA + "weight_noisy.csv", parse_dates=["timestamp"]).set_index("timestamp")["weight"]
print("samples:", s.size, "| range: %.1f .. %.1f" % (s.min(), s.max()))
s.head()

In [ ]:
from scipy.signal import medfilt

def moving_average(x, window=21):
    return x.rolling(window, center=True, min_periods=1).mean()

def ewma(x, span=21):
    return x.ewm(span=span, adjust=False).mean()

def median_filter(x, kernel=9):
    k = kernel if kernel % 2 else kernel + 1
    return pd.Series(medfilt(x.to_numpy(float), k), index=x.index)

## 1 · Overlay the three on one window

In [ ]:
w = s.iloc[3000:3600]
plt.figure(figsize=(11,4))
plt.plot(w.index, w.values, lw=.5, alpha=.5, label="raw")
plt.plot(w.index, moving_average(w, 21).values, label="moving average (21)")
plt.plot(w.index, ewma(w, 21).values, label="EWMA (span 21)")
plt.plot(w.index, median_filter(w, 9).values, label="median (9)")
plt.legend(); plt.title("Basic smoothing filters"); plt.show()

## 2 · The window trade-off
Bigger window = smoother, but it lags and rounds off the real fill edges.

In [ ]:
plt.figure(figsize=(11,3))
plt.plot(w.index, w.values, lw=.4, alpha=.3, label="raw")
for win in [5, 21, 51]:
    plt.plot(w.index, moving_average(w, win).values, label=f"window {win}")
plt.legend(); plt.title("Moving average - window trade-off"); plt.show()

## 3 · Median vs mean on spikes
A washing-fluid spike is an outlier. The mean gets dragged toward it; the median ignores it.

In [ ]:
seg = s.iloc[500:1100]
plt.figure(figsize=(11,3))
plt.plot(seg.index, seg.values, lw=.5, alpha=.5, label="raw (with spikes)")
plt.plot(seg.index, moving_average(seg, 15).values, label="moving average - smears the spike")
plt.plot(seg.index, median_filter(seg, 15).values, label="median - rejects the spike")
plt.legend(); plt.title("Median rejects spikes; moving average smears them"); plt.show()

## 4 · Score them
Residual std (noise left) and total variation (roughness). Lower = smoother.

In [ ]:
def resid_std(x):
    m = x.rolling(15, center=True, min_periods=1).median()
    return (x - m).std()

def total_variation(x):
    return float(np.abs(np.diff(x.to_numpy(float))).sum())

print(f"{'filter':14}{'resid_std':>11}{'total_var':>12}")
for name, x in [("raw", s), ("moving_avg", moving_average(s, 21)),
                ("ewma", ewma(s, 21)), ("median", median_filter(s, 9))]:
    print(f"{name:14}{resid_std(x):>11.4f}{total_variation(x):>12,.0f}")

## Debrief
- **Median** for spikes; **moving average / EWMA** for the dither; the right choice depends on the noise you saw in Filtering 1.
- Every smoother trades noise reduction against lag and edge rounding - there is no free lunch.
- Next (Filtering 3): frequency-domain filters that remove noise while preserving the fill *shape*.